# Tech Challenge - Data Engineering

## 01 - Ingestão Batch | Camada Bronze

### Objetivo

Este notebook é responsável pela ingestão dos dados públicos utilizados no Tech Challenge do módulo 2.

Os dados são disponibilizados pela plataforma **Base dos Dados** e armazenados publicamente no Google BigQuery.

A ingestão é realizada diretamente pelo Apache Spark utilizando o Spark BigQuery Connector, evitando a necessidade de download manual dos arquivos CSV.

Os dados são persistidos no Databricks no formato **Delta Lake**, compondo a camada **Bronze** da arquitetura medalhão.

### Fonte

**Dataset:** Avaliação da Alfabetização  
**Organização:** Instituto Nacional de Estudos e Pesquisas Educacionais Anísio Teixeira (INEP)  
**Dataset BigQuery:** `basedosdados.br_inep_avaliacao_alfabetizacao`

### Arquitetura

Base dos Dados / BigQuery  
→ Spark BigQuery Connector  
→ Apache Spark  
→ Delta Lake  
→ Camada Bronze

# Configuração

## Bibliotecas

In [0]:
import basedosdados as bd

from pyspark.sql import functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

## Autenticação

O acesso ao Google BigQuery é realizado por meio de uma Service Account criada exclusivamente para o projeto.

Por segurança, a chave da Service Account **não é armazenada no código-fonte nem no GitHub**.

A credencial foi armazenada utilizando o recurso **Databricks Secrets** e é recuperada em tempo de execução.

Dessa forma, o notebook pode ser versionado publicamente sem exposição de credenciais.

In [0]:
credencial_gcp = dbutils.secrets.get(
    scope="fiap-tech-challenge",
    key="gcp-service-account-b64"
)

if not credencial_gcp:
    raise ValueError("Credencial do Google Cloud não encontrada.")

print("Credencial carregada com sucesso.")

# Criação da camada Bronze

A arquitetura medalhão divide os dados em diferentes níveis de processamento.

Neste notebook será criada exclusivamente a camada Bronze.
**Bronze:** dados provenientes diretamente da fonte, com mínima transformação.

Criando o schema Bronze

In [0]:
# Criando o schema da tabela Bronze
bronze_schema = "bronze"

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {bronze_schema}
""")

print(f"Schema '{bronze_schema}' disponível.")

### Tabelas de origem

O dataset possui seis tabelas analíticas e uma tabela adicional contendo o dicionário dos dados.

As sete tabelas serão ingeridas:

- `alunos`
- `municipio`
- `uf`
- `meta_alfabetizacao_municipio`
- `meta_alfabetizacao_uf`
- `meta_alfabetizacao_brasil`
- `dicionario`

A tabela `alunos` possui granularidade por estudante e concentra a maior parte do volume do dataset.

As demais tabelas contêm indicadores agregados e metas nos níveis municipal, estadual e nacional.

In [0]:
# Identificação do projeto utilizado para autenticação e execução
# das consultas no Google BigQuery.
billing_project_id = "fiap-tech-challenge2"

# Projeto público da Base dos Dados
source_project = "basedosdados"

# Dataset de origem
source_dataset = "br_inep_avaliacao_alfabetizacao"

# Schema de destino no Databricks
bronze_schema = "bronze"

# Tabelas que serão ingeridas
tabelas = [
    "alunos",
    "municipio",
    "uf",
    "meta_alfabetizacao_municipio",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_brasil",
    "dicionario"
]

## Função de Leitura

In [0]:
def ler_tabela_bigquery(nome_tabela):
    """
    Realiza a leitura de uma tabela pública da Base dos Dados
    diretamente do Google BigQuery utilizando Spark.

    Parameters
    ----------
    nome_tabela : str
        Nome da tabela existente no dataset de origem.

    Returns
    -------
    pyspark.sql.DataFrame
        DataFrame Spark contendo os dados da tabela.
    """

    tabela_origem = (
        f"{source_project}."
        f"{source_dataset}."
        f"{nome_tabela}"
    )

    df = (
        spark.read
        .format("bigquery")
        .option("credentials", credencial_gcp)
        .option("parentProject", billing_project_id)
        .load(tabela_origem)
    )

    return df

## Ingestão Batch

A ingestão será executada para todas as tabelas configuradas anteriormente.

Para cada tabela:

1. Os dados são lidos diretamente do BigQuery.
2. Metadados técnicos de ingestão são adicionados.
3. Os dados são persistidos no formato Delta Lake.
4. A tabela é criada dentro do schema `bronze`.

A estratégia utilizada neste primeiro estágio é uma **carga completa (full load)**.

In [0]:
resultados = []

for tabela in tabelas:

    tabela_origem = f"{source_project}." f"{source_dataset}." f"{tabela}"

    tabela_destino = f"{bronze_schema}.{tabela}"

    print("=" * 70)
    print(f"Iniciando ingestão: {tabela}")
    print(f"Origem:  {tabela_origem}")
    print(f"Destino: {tabela_destino}")

    try:

        # Leitura do BigQuery
        df = ler_tabela_bigquery(tabela)

        # Inclusão de metadados técnicos
        df_bronze = (
            df.withColumn("_ingestion_timestamp", F.current_timestamp())
            .withColumn("_source_system", F.lit("Base dos Dados / BigQuery"))
            .withColumn("_source_table", F.lit(tabela_origem))
        )

        # Persistência no Delta Lake
        (
            df_bronze.write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(tabela_destino)
        )

        resultados.append((tabela, tabela_destino, "SUCESSO", None))

        print(f"✓ {tabela} carregada com sucesso.")

    except Exception as e:

        resultados.append((tabela, tabela_destino, "ERRO", str(e)[:500]))

        print(f"✗ Erro ao carregar {tabela}")
        print(str(e)[:1000])

# Validação da camada Bronze

Após a ingestão, são realizadas validações simples para garantir que todas as tabelas esperadas foram criadas corretamente.

## Status de Ingestão

In [0]:
schema_resultados = StructType([
    StructField("tabela_origem", StringType(), False),
    StructField("tabela_destino", StringType(), False),
    StructField("status", StringType(), False),
    StructField("erro", StringType(), True)
])

df_resultados = spark.createDataFrame(
    resultados,
    schema=schema_resultados
)

display(df_resultados)

Checagem do schema bronze criado, e das tabelas populadas

In [0]:
display(
    spark.sql(
        f"SHOW TABLES IN {bronze_schema}"
    )
)

## Contagem de Resgitros

In [0]:
validacao = []

for tabela in tabelas:

    tabela_destino = f"{bronze_schema}.{tabela}"

    df = spark.table(tabela_destino)

    quantidade_linhas = df.count()
    quantidade_colunas = len(df.columns)

    validacao.append(
        (
            tabela,
            quantidade_linhas,
            quantidade_colunas
        )
    )

df_validacao = spark.createDataFrame(
    validacao,
    [
        "tabela",
        "quantidade_linhas",
        "quantidade_colunas"
    ]
)

display(df_validacao)

## Inspecionando
Analisando amostras e tipos de dados da tabela "alunos", que é a maior

In [0]:
display(
    spark.table("bronze.alunos")
    .limit(10)
)

In [0]:
spark.table(
    "bronze.alunos"
).printSchema()